# YOLO11n-P2 — Stage 4.1 Hybrid Recovery (Standalone)

This notebook is **fully standalone**. It does not depend on variables left in RAM by any previous notebook.

## Starting point

It continues from the already-trained Stage-4 P2 checkpoint:

`YOLO11n_P2_Stage4_SmallPriority/Detection_Stage4_P2/weights/best.pt`

## Strategy

**Stage-4 P2 best.pt → rebuild the same Small-Priority train dataset → +20 gentle recovery epochs → audit every saved checkpoint → compare against Stage 2 / Stage 3 / Stage 4.**

The recovery recipe combines the useful ideas from previous YOLO experiments:

- keep **P2/P3/P4/P5** architecture unchanged
- fixed **960 px**
- Stage-4 Small-Priority sliced data
- low learning rate
- low Mosaic
- mild underwater/color robustness
- no blur / downscale / JPEG / strong noise
- original validation split only
- **Detector test is not used**


In [ ]:
# ============================================================
# CELL 1 — INSTALL + MOUNT + ENVIRONMENT
# ============================================================

!pip -q install ultralytics==8.4.117 albumentations==2.0.8 pycocotools pyyaml

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import defaultdict

import os
import json
import random
import shutil
import time
import math
import yaml

import numpy as np
import pandas as pd
import cv2
import torch
import albumentations as A
import ultralytics

from ultralytics import YOLO

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Colab before continuing.")

DEVICE = 0
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print("=" * 110)
print("ENVIRONMENT")
print("=" * 110)
print("Ultralytics    :", ultralytics.__version__)
print("Albumentations :", A.__version__)
print("PyTorch        :", torch.__version__)
print("GPU            :", GPU_NAME)
print(f"VRAM           : {VRAM_GB:.2f} GB")
print("STATUS: ENVIRONMENT_READY")


In [ ]:
# ============================================================
# CELL 2 — PROJECT PATHS + CHECKPOINT CHECK
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Coral_Reef_AI_Project"
)

DETECTION_V1 = (
    PROJECT_ROOT
    / "01_Data"
    / "Final"
    / "Detection_v1"
)

TRAIN_IMG_DIR = DETECTION_V1 / "images" / "train"
VAL_IMG_DIR = DETECTION_V1 / "images" / "validation"

TRAIN_COCO = DETECTION_V1 / "annotations" / "instances_train.json"
VAL_COCO = DETECTION_V1 / "annotations" / "instances_validation.json"

EXP_ROOT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_P2_Stage4_SmallPriority"
)

BEST_STAGE4 = (
    EXP_ROOT
    / "Detection_Stage4_P2"
    / "weights"
    / "best.pt"
)

# New Stage-4.1 output folder.
STAGE41_ROOT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_P2_Stage4_1_HybridRecovery"
)

STAGE41_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# Local runtime dataset. Rebuilt every fresh Colab session.
BUILD_ROOT = Path(
    "/content/Coral_YOLO11_P2_Stage41_SmallPriority"
)

DATA_YAML = BUILD_ROOT / "data.yaml"

required_paths = [
    TRAIN_IMG_DIR,
    VAL_IMG_DIR,
    TRAIN_COCO,
    VAL_COCO,
    BEST_STAGE4,
]

for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(p)

with open(TRAIN_COCO, "r", encoding="utf-8") as f:
    train_coco = json.load(f)

with open(VAL_COCO, "r", encoding="utf-8") as f:
    val_coco = json.load(f)

print("=" * 110)
print("SOURCE CHECK")
print("=" * 110)
print("Stage4 best :", BEST_STAGE4)
print("Train       :", len(train_coco["images"]), "images /", len(train_coco["annotations"]), "boxes")
print("Validation  :", len(val_coco["images"]), "images /", len(val_coco["annotations"]), "boxes")

assert len(train_coco["images"]) == 452
assert len(train_coco["annotations"]) == 1540
assert len(val_coco["images"]) == 100
assert len(val_coco["annotations"]) == 354

print("\nDetector TEST was not loaded.")
print("STATUS: SOURCE_CHECK_PASSED")


In [ ]:
# ============================================================
# CELL 3 — REBUILD THE SAME SMALL-PRIORITY STAGE-4 DATASET
#
# Train only:
#   452 original train images
#   + true-small-centered slices
#   + crowded positive slices
#   + limited other positives
#   + limited negatives
#
# Validation:
#   original 100 images only
# ============================================================

SMALL_AREA_RATIO = 0.01
MEDIUM_AREA_RATIO = 0.10
CROWDED_GT_COUNT = 4

SLICE_SIZE = 640
GRID_OVERLAP = 0.20
MIN_VISIBLE_FRACTION = 0.40

MAX_SMALL_SLICES = 350
MAX_CROWDED_SLICES = 150
MAX_OTHER_POSITIVE_SLICES = 50
MAX_NEGATIVE_SLICES = 60

CENTER_JITTERS = [
    (0.00, 0.00),
    (0.12, 0.00),
    (-0.12, 0.00),
]

rng = random.Random(SEED)

train_images_by_id = {
    int(im["id"]): im
    for im in train_coco["images"]
}

anns_by_image = defaultdict(list)

for ann in train_coco["annotations"]:
    anns_by_image[int(ann["image_id"])].append(ann)


def clamp(v, lo, hi):
    return max(lo, min(v, hi))


def make_centered_origin(cx, cy, W, H, tile_size, jx=0.0, jy=0.0):
    tw = min(tile_size, W)
    th = min(tile_size, H)

    target_cx = cx + jx * tw
    target_cy = cy + jy * th

    x0 = int(round(target_cx - tw / 2))
    y0 = int(round(target_cy - th / 2))

    x0 = int(clamp(x0, 0, max(W - tw, 0)))
    y0 = int(clamp(y0, 0, max(H - th, 0)))

    return x0, y0, int(tw), int(th)


def grid_starts(length, tile, overlap):
    if length <= tile:
        return [0]

    step = max(1, int(round(tile * (1.0 - overlap))))
    starts = list(range(0, max(length - tile + 1, 1), step))
    last = length - tile

    if starts[-1] != last:
        starts.append(last)

    return sorted(set(starts))


def clip_ann_to_tile(ann, tile_x, tile_y, tile_w, tile_h):
    x, y, bw, bh = map(float, ann["bbox"])

    x1, y1 = x, y
    x2, y2 = x + bw, y + bh

    tx1, ty1 = float(tile_x), float(tile_y)
    tx2, ty2 = float(tile_x + tile_w), float(tile_y + tile_h)

    cx1 = max(x1, tx1)
    cy1 = max(y1, ty1)
    cx2 = min(x2, tx2)
    cy2 = min(y2, ty2)

    clipped_w = max(0.0, cx2 - cx1)
    clipped_h = max(0.0, cy2 - cy1)

    original_area = max(bw * bh, 1e-9)
    visible_area = clipped_w * clipped_h
    visible_fraction = visible_area / original_area

    if (
        clipped_w <= 1.0
        or clipped_h <= 1.0
        or visible_fraction < MIN_VISIBLE_FRACTION
    ):
        return None

    return {
        "annotation_id": int(ann["id"]),
        "x": cx1 - tx1,
        "y": cy1 - ty1,
        "w": clipped_w,
        "h": clipped_h,
        "visible_fraction": visible_fraction,
    }


def candidate_from_origin(image_id, x0, y0, tw, th, reason):
    im = train_images_by_id[image_id]

    W = float(im["width"])
    H = float(im["height"])
    image_area = max(W * H, 1.0)

    kept = []

    for ann in anns_by_image[image_id]:
        clipped = clip_ann_to_tile(
            ann,
            x0,
            y0,
            tw,
            th,
        )

        if clipped is None:
            continue

        _, _, bw, bh = map(float, ann["bbox"])
        original_rel_area = (bw * bh) / image_area

        clipped["original_relative_area"] = original_rel_area
        clipped["original_small"] = bool(
            original_rel_area < SMALL_AREA_RATIO
        )

        kept.append(clipped)

    return {
        "image_id": int(image_id),
        "file_name": Path(im["file_name"]).name,
        "x0": int(x0),
        "y0": int(y0),
        "tile_w": int(tw),
        "tile_h": int(th),
        "reason": reason,
        "annotations": kept,
        "contains_small": any(x["original_small"] for x in kept),
        "positive": len(kept) > 0,
        "kept_box_count": len(kept),
        "crowded_source": len(anns_by_image[image_id]) >= CROWDED_GT_COUNT,
    }


candidate_map = {}


def add_candidate(c):
    key = (
        c["image_id"],
        c["x0"],
        c["y0"],
        c["tile_w"],
        c["tile_h"],
    )

    if key not in candidate_map:
        candidate_map[key] = c
    else:
        old_reason = candidate_map[key]["reason"]
        if c["reason"] not in old_reason.split("+"):
            candidate_map[key]["reason"] = old_reason + "+" + c["reason"]


# ------------------------------------------------------------
# A) TRUE SMALL OBJECT-CENTERED SLICES
# ------------------------------------------------------------

small_gt_count = 0

for image_id, anns in anns_by_image.items():
    im = train_images_by_id[image_id]

    W = int(im["width"])
    H = int(im["height"])
    image_area = max(float(W * H), 1.0)

    for ann in anns:
        x, y, bw, bh = map(float, ann["bbox"])
        rel_area = (bw * bh) / image_area

        if rel_area >= SMALL_AREA_RATIO:
            continue

        small_gt_count += 1

        cx = x + bw / 2.0
        cy = y + bh / 2.0

        for jx, jy in CENTER_JITTERS:
            x0, y0, tw, th = make_centered_origin(
                cx, cy, W, H, SLICE_SIZE, jx, jy
            )

            add_candidate(
                candidate_from_origin(
                    image_id,
                    x0,
                    y0,
                    tw,
                    th,
                    "small_centered",
                )
            )


# ------------------------------------------------------------
# B) GRID SLICES FOR CROWDED TRAIN SOURCES
# ------------------------------------------------------------

crowded_ids = {
    image_id
    for image_id, anns in anns_by_image.items()
    if len(anns) >= CROWDED_GT_COUNT
}

for image_id in sorted(crowded_ids):
    im = train_images_by_id[image_id]

    W = int(im["width"])
    H = int(im["height"])

    tw = min(SLICE_SIZE, W)
    th = min(SLICE_SIZE, H)

    for y0 in grid_starts(H, th, GRID_OVERLAP):
        for x0 in grid_starts(W, tw, GRID_OVERLAP):
            add_candidate(
                candidate_from_origin(
                    image_id,
                    x0,
                    y0,
                    tw,
                    th,
                    "crowded_grid",
                )
            )


candidates = list(candidate_map.values())

small_candidates = [
    c for c in candidates
    if c["contains_small"]
]

crowded_positive_candidates = [
    c for c in candidates
    if (
        not c["contains_small"]
        and c["crowded_source"]
        and c["positive"]
        and c["kept_box_count"] >= 2
    )
]

crowded_keys = {
    (
        c["image_id"],
        c["x0"],
        c["y0"],
        c["tile_w"],
        c["tile_h"],
    )
    for c in crowded_positive_candidates
}

other_positive_candidates = [
    c for c in candidates
    if (
        not c["contains_small"]
        and c["positive"]
        and (
            c["image_id"],
            c["x0"],
            c["y0"],
            c["tile_w"],
            c["tile_h"],
        ) not in crowded_keys
    )
]

negative_candidates = [
    c for c in candidates
    if not c["positive"]
]


def choose(items, max_n):
    if len(items) <= max_n:
        return list(items)
    return rng.sample(items, max_n)


chosen_small = choose(
    small_candidates,
    MAX_SMALL_SLICES,
)

chosen_crowded = choose(
    crowded_positive_candidates,
    MAX_CROWDED_SLICES,
)

chosen_other = choose(
    other_positive_candidates,
    MAX_OTHER_POSITIVE_SLICES,
)

positive_count = (
    len(chosen_small)
    + len(chosen_crowded)
    + len(chosen_other)
)

negative_cap = min(
    MAX_NEGATIVE_SLICES,
    max(1, int(round(positive_count * 0.12))),
)

chosen_negative = choose(
    negative_candidates,
    negative_cap,
)

selected_candidates = (
    chosen_small
    + chosen_crowded
    + chosen_other
    + chosen_negative
)

selected_unique = {}

for c in selected_candidates:
    key = (
        c["image_id"],
        c["x0"],
        c["y0"],
        c["tile_w"],
        c["tile_h"],
    )
    selected_unique[key] = c

selected_candidates = list(selected_unique.values())

selected_small_occurrences = sum(
    sum(a["original_small"] for a in c["annotations"])
    for c in selected_candidates
)

print("=" * 110)
print("SMALL-PRIORITY SELECTION")
print("=" * 110)

print("Original train small GT       :", small_gt_count)
print("All unique candidates         :", len(candidates))
print("Small-containing available    :", len(small_candidates))
print("Crowded-positive available    :", len(crowded_positive_candidates))
print("Other-positive available      :", len(other_positive_candidates))
print("Negative available            :", len(negative_candidates))

print("\nSELECTED")
print("Small-containing slices       :", len(chosen_small))
print("Crowded-positive slices       :", len(chosen_crowded))
print("Other-positive slices         :", len(chosen_other))
print("Negative slices               :", len(chosen_negative))
print("Final unique slices           :", len(selected_candidates))
print("Small-GT occurrences in slices:", selected_small_occurrences)

assert len(chosen_small) > 0
assert len(selected_candidates) > 0

print("\nSTATUS: SMALL_PRIORITY_SELECTION_READY")


In [ ]:
# ============================================================
# CELL 4 — MATERIALIZE DATASET + STRICT VALIDATION
# ============================================================

if BUILD_ROOT.exists():
    shutil.rmtree(BUILD_ROOT)

for split in ["train", "val"]:
    (BUILD_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (BUILD_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)


def safe_symlink(src, dst):
    src = Path(src)
    dst = Path(dst)

    if dst.exists() or dst.is_symlink():
        dst.unlink()

    os.symlink(str(src), str(dst))


def coco_original_to_yolo(
    coco,
    source_img_dir,
    out_img_dir,
    out_label_dir,
    prefix,
):
    local_anns = defaultdict(list)

    for ann in coco["annotations"]:
        local_anns[int(ann["image_id"])].append(ann)

    image_count = 0
    box_count = 0

    for im in coco["images"]:
        image_id = int(im["id"])
        src_name = Path(im["file_name"]).name
        src = Path(source_img_dir) / src_name

        if not src.exists():
            raise FileNotFoundError(src)

        dst_name = f"{prefix}{src_name}"

        safe_symlink(
            src,
            Path(out_img_dir) / dst_name,
        )

        W = float(im["width"])
        H = float(im["height"])

        lines = []

        for ann in local_anns[image_id]:
            x, y, bw, bh = map(float, ann["bbox"])

            xc = (x + bw / 2.0) / W
            yc = (y + bh / 2.0) / H
            nw = bw / W
            nh = bh / H

            lines.append(
                f"0 {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}"
            )

            box_count += 1

        label_path = (
            Path(out_label_dir)
            / f"{Path(dst_name).stem}.txt"
        )

        label_path.write_text(
            "\n".join(lines) + ("\n" if lines else ""),
            encoding="utf-8",
        )

        image_count += 1

    return image_count, box_count


# Original train.
orig_train_images, orig_train_boxes = coco_original_to_yolo(
    train_coco,
    TRAIN_IMG_DIR,
    BUILD_ROOT / "images" / "train",
    BUILD_ROOT / "labels" / "train",
    "orig__",
)

# Original validation.
val_images, val_boxes = coco_original_to_yolo(
    val_coco,
    VAL_IMG_DIR,
    BUILD_ROOT / "images" / "val",
    BUILD_ROOT / "labels" / "val",
    "val__",
)


# Materialize selected slices.
slice_rows = []
slice_total_boxes = 0
slice_small_occurrences = 0

for idx, c in enumerate(selected_candidates):
    src = TRAIN_IMG_DIR / c["file_name"]

    image = cv2.imread(
        str(src),
        cv2.IMREAD_COLOR,
    )

    if image is None:
        raise RuntimeError(f"Cannot read {src}")

    x0 = c["x0"]
    y0 = c["y0"]
    tw = c["tile_w"]
    th = c["tile_h"]

    crop = image[
        y0:y0 + th,
        x0:x0 + tw,
    ]

    if crop.size == 0:
        raise RuntimeError(f"Empty crop: {c}")

    slice_name = (
        f"slice_{idx:04d}"
        f"__img{c['image_id']}"
        f"__x{x0}_y{y0}"
        f".jpg"
    )

    image_path = (
        BUILD_ROOT
        / "images"
        / "train"
        / slice_name
    )

    ok = cv2.imwrite(
        str(image_path),
        crop,
        [int(cv2.IMWRITE_JPEG_QUALITY), 95],
    )

    if not ok:
        raise RuntimeError(f"Failed writing {image_path}")

    lines = []

    for ann in c["annotations"]:
        xc = (ann["x"] + ann["w"] / 2.0) / tw
        yc = (ann["y"] + ann["h"] / 2.0) / th
        nw = ann["w"] / tw
        nh = ann["h"] / th

        lines.append(
            f"0 {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}"
        )

        slice_total_boxes += 1

        if ann["original_small"]:
            slice_small_occurrences += 1

    label_path = (
        BUILD_ROOT
        / "labels"
        / "train"
        / f"{Path(slice_name).stem}.txt"
    )

    label_path.write_text(
        "\n".join(lines) + ("\n" if lines else ""),
        encoding="utf-8",
    )

    slice_rows.append({
        "slice_name": slice_name,
        "source_image_id": c["image_id"],
        "source_file_name": c["file_name"],
        "x0": x0,
        "y0": y0,
        "tile_w": tw,
        "tile_h": th,
        "reason": c["reason"],
        "positive": c["positive"],
        "box_count": c["kept_box_count"],
        "contains_small": c["contains_small"],
        "small_box_occurrences": int(
            sum(a["original_small"] for a in c["annotations"])
        ),
        "crowded_source": c["crowded_source"],
    })


slice_manifest_df = pd.DataFrame(slice_rows)

SLICE_MANIFEST = (
    STAGE41_ROOT
    / "stage41_small_priority_slice_manifest.csv"
)

slice_manifest_df.to_csv(
    SLICE_MANIFEST,
    index=False,
)


# data.yaml
data_yaml = {
    "path": str(BUILD_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "target_coral"},
}

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        data_yaml,
        f,
        sort_keys=False,
    )


def count_yolo_boxes(label_dir):
    total = 0
    invalid = []

    for lp in Path(label_dir).glob("*.txt"):
        txt = lp.read_text(encoding="utf-8").strip()

        if not txt:
            continue

        for line in txt.splitlines():
            parts = line.split()

            if len(parts) != 5:
                invalid.append((lp.name, line))
                continue

            cls, xc, yc, w, h = map(float, parts)

            valid = (
                int(cls) == 0
                and 0 <= xc <= 1
                and 0 <= yc <= 1
                and 0 < w <= 1
                and 0 < h <= 1
            )

            if not valid:
                invalid.append((lp.name, line))

            total += 1

    return total, invalid


train_box_count, train_invalid = count_yolo_boxes(
    BUILD_ROOT / "labels" / "train"
)

val_box_count, val_invalid = count_yolo_boxes(
    BUILD_ROOT / "labels" / "val"
)

train_imgs = list(
    (BUILD_ROOT / "images" / "train").glob("*")
)

val_imgs = list(
    (BUILD_ROOT / "images" / "val").glob("*")
)

assert not train_invalid, train_invalid[:10]
assert not val_invalid, val_invalid[:10]

assert orig_train_images == 452
assert orig_train_boxes == 1540
assert val_images == 100
assert val_boxes == 354

assert train_box_count == 1540 + slice_total_boxes
assert val_box_count == 354

assert len(val_imgs) == 100
assert all(p.name.startswith("val__") for p in val_imgs)

print("=" * 110)
print("STAGE-4.1 DATASET READY")
print("=" * 110)
print("Original train images :", orig_train_images)
print("Original train boxes  :", orig_train_boxes)
print("Added slices          :", len(slice_manifest_df))
print("Added slice boxes     :", slice_total_boxes)
print("Small box occurrences :", slice_small_occurrences)
print("Final train images    :", len(train_imgs))
print("Final train boxes     :", train_box_count)
print("Validation images     :", len(val_imgs))
print("Validation boxes      :", val_box_count)
print("Data YAML             :", DATA_YAML)
print("\nTEST IS NOT PRESENT.")
print("STATUS: DATASET_VALIDATED")


In [ ]:
# ============================================================
# CELL 5 — VALIDATION AUDIT FUNCTIONS
# ============================================================

val_image_id_by_basename = {
    Path(im["file_name"]).name: int(im["id"])
    for im in val_coco["images"]
}

val_image_meta = {
    int(im["id"]): im
    for im in val_coco["images"]
}

val_gt_by_id = defaultdict(list)

for ann in val_coco["annotations"]:
    image_id = int(ann["image_id"])
    x, y, w, h = map(float, ann["bbox"])

    val_gt_by_id[image_id].append({
        "bbox": np.array(
            [x, y, x + w, y + h],
            dtype=np.float32,
        ),
        "area": float(w * h),
    })


original_val_paths = sorted([
    p
    for p in VAL_IMG_DIR.iterdir()
    if p.suffix.lower()
    in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
])

assert len(original_val_paths) == 100


def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros(
            (len(a), len(b)),
            dtype=np.float32,
        )

    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)

    lt = np.maximum(
        a[:, None, :2],
        b[None, :, :2],
    )

    rb = np.minimum(
        a[:, None, 2:],
        b[None, :, 2:],
    )

    wh = np.clip(
        rb - lt,
        0,
        None,
    )

    inter = wh[..., 0] * wh[..., 1]

    aa = (
        np.clip(a[:, 2] - a[:, 0], 0, None)
        * np.clip(a[:, 3] - a[:, 1], 0, None)
    )

    bb = (
        np.clip(b[:, 2] - b[:, 0], 0, None)
        * np.clip(b[:, 3] - b[:, 1], 0, None)
    )

    union = aa[:, None] + bb[None, :] - inter

    return inter / np.clip(
        union,
        1e-9,
        None,
    )


def greedy_match(pred_boxes, gt_boxes, threshold=0.50):
    if len(pred_boxes) == 0:
        return [], [], list(range(len(gt_boxes)))

    if len(gt_boxes) == 0:
        return [], list(range(len(pred_boxes))), []

    ious = iou_matrix(
        pred_boxes,
        gt_boxes,
    )

    candidates = []

    for p in range(len(pred_boxes)):
        for g in range(len(gt_boxes)):
            if ious[p, g] >= threshold:
                candidates.append(
                    (float(ious[p, g]), p, g)
                )

    candidates.sort(reverse=True)

    used_p = set()
    used_g = set()
    matches = []

    for iou, p, g in candidates:
        if p in used_p or g in used_g:
            continue

        used_p.add(p)
        used_g.add(g)
        matches.append((p, g, iou))

    unmatched_p = [
        p for p in range(len(pred_boxes))
        if p not in used_p
    ]

    unmatched_g = [
        g for g in range(len(gt_boxes))
        if g not in used_g
    ]

    return matches, unmatched_p, unmatched_g


def size_bucket(gt_box, image_w, image_h):
    x1, y1, x2, y2 = map(float, gt_box)

    ratio = (
        max(0.0, x2 - x1)
        * max(0.0, y2 - y1)
        / max(float(image_w * image_h), 1.0)
    )

    if ratio < 0.01:
        return "small"

    if ratio < 0.10:
        return "medium"

    return "large"


def threshold_sweep(pred_dict):
    rows = []

    for conf in np.round(
        np.arange(0.05, 0.81, 0.025),
        3,
    ):
        TP = FP = FN = 0

        for image_name, pred in pred_dict.items():
            image_id = val_image_id_by_basename[image_name]

            gt_boxes = np.array(
                [
                    x["bbox"]
                    for x in val_gt_by_id[image_id]
                ],
                dtype=np.float32,
            ).reshape(-1, 4)

            keep = pred["conf"] >= conf
            pred_boxes = pred["boxes"][keep]

            matches, up, ug = greedy_match(
                pred_boxes,
                gt_boxes,
                0.50,
            )

            TP += len(matches)
            FP += len(up)
            FN += len(ug)

        precision = (
            TP / (TP + FP)
            if (TP + FP)
            else 0.0
        )

        recall = (
            TP / (TP + FN)
            if (TP + FN)
            else 0.0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        rows.append({
            "confidence": float(conf),
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

    df = pd.DataFrame(rows)
    best = df.loc[df["f1"].idxmax()]

    return df, best


def detailed_metrics(pred_dict, conf):
    gt_rows = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    crowded_totals = {
        False: {"TP": 0, "FP": 0, "FN": 0, "GT": 0, "images": 0},
        True: {"TP": 0, "FP": 0, "FN": 0, "GT": 0, "images": 0},
    }

    for image_name, pred in pred_dict.items():
        image_id = val_image_id_by_basename[image_name]
        meta = val_image_meta[image_id]
        gt_items = val_gt_by_id[image_id]

        gt_boxes = np.array(
            [x["bbox"] for x in gt_items],
            dtype=np.float32,
        ).reshape(-1, 4)

        keep = pred["conf"] >= conf
        pred_boxes = pred["boxes"][keep]

        matches, up, ug = greedy_match(
            pred_boxes,
            gt_boxes,
            0.50,
        )

        matched_gt = {
            g: (p, iou)
            for p, g, iou in matches
        }

        for g, gt in enumerate(gt_items):
            bucket = size_bucket(
                gt["bbox"],
                int(meta["width"]),
                int(meta["height"]),
            )

            gt_rows.append({
                "image": image_name,
                "size": bucket,
                "detected": g in matched_gt,
                "iou": (
                    float(matched_gt[g][1])
                    if g in matched_gt
                    else 0.0
                ),
            })

        tp = len(matches)
        fp = len(up)
        fn = len(ug)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        crowded = len(gt_items) >= CROWDED_GT_COUNT

        crowded_totals[crowded]["TP"] += tp
        crowded_totals[crowded]["FP"] += fp
        crowded_totals[crowded]["FN"] += fn
        crowded_totals[crowded]["GT"] += len(gt_items)
        crowded_totals[crowded]["images"] += 1

    gt_df = pd.DataFrame(gt_rows)

    size_rows = []

    for bucket in ["small", "medium", "large"]:
        g = gt_df[gt_df["size"] == bucket]

        size_rows.append({
            "size": bucket,
            "GT": len(g),
            "TP": int(g["detected"].sum()),
            "FN": int((~g["detected"]).sum()),
            "recall": (
                float(g["detected"].mean())
                if len(g)
                else np.nan
            ),
            "mean_matched_iou": (
                float(g.loc[g["detected"], "iou"].mean())
                if g["detected"].any()
                else np.nan
            ),
        })

    crowded_rows = []

    for crowded in [False, True]:
        d = crowded_totals[crowded]

        precision = (
            d["TP"] / (d["TP"] + d["FP"])
            if (d["TP"] + d["FP"])
            else 0.0
        )

        recall = (
            d["TP"] / (d["TP"] + d["FN"])
            if (d["TP"] + d["FN"])
            else 0.0
        )

        crowded_rows.append({
            "crowded": crowded,
            "images": d["images"],
            "GT": d["GT"],
            "TP": d["TP"],
            "FP": d["FP"],
            "FN": d["FN"],
            "precision": precision,
            "recall": recall,
        })

    p_all = (
        total_tp / (total_tp + total_fp)
        if (total_tp + total_fp)
        else 0.0
    )

    r_all = (
        total_tp / (total_tp + total_fn)
        if (total_tp + total_fn)
        else 0.0
    )

    f1_all = (
        2 * p_all * r_all / (p_all + r_all)
        if (p_all + r_all)
        else 0.0
    )

    overall = {
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
        "precision": p_all,
        "recall": r_all,
        "f1": f1_all,
    }

    return (
        overall,
        pd.DataFrame(size_rows),
        pd.DataFrame(crowded_rows),
        gt_df,
    )


def value_for_size(size_df, bucket):
    return float(
        size_df.loc[
            size_df["size"] == bucket,
            "recall",
        ].iloc[0]
    )


print("Original validation images:", len(original_val_paths))
print("STATUS: AUDIT_FUNCTIONS_READY")


In [ ]:
# ============================================================
# CELL 6 — PREVIOUS YOLO REFERENCE METRICS
#
# Same validation protocol / IoU>=0.5 object matching.
# Detector test is still untouched.
# ============================================================

stage2_ref = {
    "method": "Stage2 Standard",
    "precision": 0.7884615385,
    "overall_recall": 0.6949152542,
    "small_recall": 0.3611111111,
    "medium_recall": 0.6745283019,
    "large_recall": 0.8490566038,
    "crowded_recall": 0.6567796610,
    "TP": 246,
    "FP": 66,
    "FN": 108,
    "best_conf": 0.325,
    "map50": 0.7727,
    "map50_95": 0.4439,
}

stage3_ref = {
    "method": "Stage3 Standard",
    "precision": 0.7937,
    "overall_recall": 0.7175,
    "small_recall": 0.4167,
    "medium_recall": 0.6981,
    "large_recall": 0.8585,
    "crowded_recall": 0.6822,
    "TP": 254,
    "FP": 66,
    "FN": 100,
    "best_conf": 0.350,
    "map50": 0.7930,
    "map50_95": 0.4730,
}

stage4_ref = {
    "method": "Stage4 YOLO11n-P2",
    "precision": 0.6319,
    "overall_recall": 0.6158,
    "small_recall": 0.2500,
    "medium_recall": 0.6132,
    "large_recall": 0.7453,
    "crowded_recall": 0.6271,
    "TP": 218,
    "FP": 127,
    "FN": 136,
    "best_conf": 0.325,
    "map50": 0.5947,
    "map50_95": 0.2628,
}

previous_df = pd.DataFrame([
    stage2_ref,
    stage3_ref,
    stage4_ref,
])

print("=" * 145)
print("PREVIOUS YOLO RESULTS")
print("=" * 145)

print(
    previous_df.to_string(
        index=False,
        formatters={
            "precision": "{:.4f}".format,
            "overall_recall": "{:.4f}".format,
            "small_recall": "{:.4f}".format,
            "medium_recall": "{:.4f}".format,
            "large_recall": "{:.4f}".format,
            "crowded_recall": "{:.4f}".format,
            "best_conf": "{:.3f}".format,
            "map50": "{:.4f}".format,
            "map50_95": "{:.4f}".format,
        },
    )
)

print("\nCurrent best standard YOLO: Stage3")
print("STATUS: REFERENCES_READY")


In [ ]:
# ============================================================
# CELL 7 — LOAD STAGE4 P2 + DEFINE HYBRID RECOVERY RECIPE
# ============================================================

STAGE41_EPOCHS = 20
STAGE41_IMGSZ = 960
STAGE41_BATCH = 6

STAGE41_LR0 = 5e-5
STAGE41_LRF = 0.08
STAGE41_WEIGHT_DECAY = 5e-4

STAGE41_NAME = "Detection_Stage4_1_HybridRecovery"

stage41_augmentations = [
    A.RGBShift(
        r_shift_limit=(-12, -2),
        g_shift_limit=(-2, 8),
        b_shift_limit=(2, 14),
        p=0.15,
    ),

    A.RandomBrightnessContrast(
        brightness_limit=(-0.12, 0.12),
        contrast_limit=(-0.14, 0.14),
        p=0.18,
    ),

    A.RandomGamma(
        gamma_limit=(85, 115),
        p=0.10,
    ),

    A.CLAHE(
        clip_limit=(1.0, 1.8),
        tile_grid_size=(8, 8),
        p=0.04,
    ),
]


stage41_model = YOLO(
    str(BEST_STAGE4)
)

stage41_strides = [
    int(round(float(x)))
    for x in stage41_model.model.stride.detach().cpu().tolist()
]

assert stage41_strides == [4, 8, 16, 32], (
    f"Expected P2/P3/P4/P5 strides [4,8,16,32], got {stage41_strides}"
)

print("=" * 110)
print("STAGE 4.1 CONFIG")
print("=" * 110)
print("Checkpoint        :", BEST_STAGE4)
print("Strides           :", stage41_strides)
print("Additional epochs :", STAGE41_EPOCHS)
print("Image size        :", STAGE41_IMGSZ)
print("Batch             :", STAGE41_BATCH)
print("Optimizer         : AdamW")
print("LR0               :", STAGE41_LR0)
print("LRF               :", STAGE41_LRF)
print("Mosaic            : 0.10")
print("Close Mosaic      : final 5 epochs")
print("Multi-scale       : OFF")
print("Blur              : OFF")
print("Downscale         : OFF")
print("JPEG Compression  : OFF")
print("Strong Noise      : OFF")
print("Test              : NOT USED")
print("STATUS: STAGE41_CONFIG_READY")


In [ ]:
# ============================================================
# CELL 8 — TRAIN STAGE 4.1 (+20 EPOCHS)
#
# This is NOT resume=True.
# We intentionally load Stage4 best weights and start a new
# optimizer/scheduler with a lower LR.
# ============================================================

print("=" * 120)
print("STARTING STAGE 4.1 — P2 HYBRID RECOVERY")
print("=" * 120)

start_stage41 = time.time()

train41 = stage41_model.train(
    data=str(DATA_YAML),

    epochs=STAGE41_EPOCHS,
    imgsz=STAGE41_IMGSZ,
    batch=STAGE41_BATCH,

    device=DEVICE,
    workers=2,

    optimizer="AdamW",
    lr0=STAGE41_LR0,
    lrf=STAGE41_LRF,
    weight_decay=STAGE41_WEIGHT_DECAY,
    cos_lr=True,
    warmup_epochs=0.5,

    multi_scale=0.0,

    degrees=5.0,
    translate=0.05,
    scale=0.15,
    shear=0.0,
    perspective=0.0,

    fliplr=0.50,
    flipud=0.05,

    mosaic=0.10,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    close_mosaic=5,

    hsv_h=0.005,
    hsv_s=0.15,
    hsv_v=0.12,

    augmentations=stage41_augmentations,

    box=7.5,
    cls=0.5,
    dfl=1.5,

    amp=True,

    seed=SEED,
    deterministic=True,

    save=True,
    save_period=2,
    patience=100,
    plots=True,

    project=str(STAGE41_ROOT),
    name=STAGE41_NAME,
    exist_ok=True,

    cache=False,
    verbose=True,
)

stage41_minutes = (
    time.time() - start_stage41
) / 60.0

RUN41 = Path(train41.save_dir)
WEIGHTS41 = RUN41 / "weights"

BEST41_ULTRA = WEIGHTS41 / "best.pt"
LAST41 = WEIGHTS41 / "last.pt"

assert BEST41_ULTRA.exists()
assert LAST41.exists()

print("\n" + "=" * 110)
print("STAGE 4.1 TRAINING COMPLETE")
print("=" * 110)
print(f"Training time : {stage41_minutes:.2f} min")
print("Run directory :", RUN41)
print("best.pt       :", BEST41_ULTRA)
print("last.pt       :", LAST41)
print("\nDetector TEST was not evaluated.")
print("STATUS: STAGE41_TRAINING_COMPLETE")


In [ ]:
# ============================================================
# CELL 9 — AUDIT EVERY SAVED STAGE-4.1 CHECKPOINT
#
# Select independently:
# 1) best mAP50:95
# 2) best operational F1
# 3) best Small Recall
# ============================================================

ANALYSIS41 = STAGE41_ROOT / "Checkpoint_Audit"
ANALYSIS41.mkdir(parents=True, exist_ok=True)


candidate_ckpts = sorted(
    WEIGHTS41.glob("epoch*.pt")
)

candidate_ckpts.extend([
    BEST41_ULTRA,
    LAST41,
])

unique_ckpts = []
seen = set()

for p in candidate_ckpts:
    p = Path(p)

    if not p.exists():
        continue

    key = str(p.resolve())

    if key not in seen:
        unique_ckpts.append(p)
        seen.add(key)

print("=" * 110)
print("CHECKPOINTS TO AUDIT")
print("=" * 110)

for p in unique_ckpts:
    print(p.name)

print("Total:", len(unique_ckpts))


def prediction_dict_for_model(model, image_paths, conf=0.05):
    start = time.time()

    results = model.predict(
        source=[str(p) for p in image_paths],
        imgsz=960,
        conf=conf,
        iou=0.70,
        max_det=300,
        device=DEVICE,
        verbose=False,
    )

    wall_time = time.time() - start

    pred_dict = {}
    inference_ms = []

    for image_path, result in zip(image_paths, results):
        if result.boxes is None or len(result.boxes) == 0:
            boxes = np.empty(
                (0, 4),
                dtype=np.float32,
            )

            confs = np.empty(
                (0,),
                dtype=np.float32,
            )

        else:
            boxes = (
                result.boxes.xyxy
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            confs = (
                result.boxes.conf
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

        pred_dict[image_path.name] = {
            "boxes": boxes,
            "conf": confs,
        }

        if (
            hasattr(result, "speed")
            and result.speed
            and "inference" in result.speed
        ):
            inference_ms.append(
                float(result.speed["inference"])
            )

    mean_model_ms = (
        float(np.mean(inference_ms))
        if inference_ms
        else None
    )

    wall_ms = (
        wall_time / len(image_paths) * 1000.0
    )

    return pred_dict, mean_model_ms, wall_ms


def audit_checkpoint(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)

    print("\n" + "-" * 115)
    print("AUDITING:", checkpoint_path.name)
    print("-" * 115)

    model = YOLO(
        str(checkpoint_path)
    )

    ultra = model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=960,
        batch=STAGE41_BATCH,
        device=DEVICE,
        conf=0.001,
        iou=0.70,
        max_det=300,
        plots=False,
        verbose=False,
    )

    map50 = float(ultra.box.map50)
    map50_95 = float(ultra.box.map)

    pred_dict, model_ms, wall_ms = prediction_dict_for_model(
        model,
        original_val_paths,
        conf=0.05,
    )

    sweep_df, best_conf_row = threshold_sweep(
        pred_dict
    )

    best_conf = float(
        best_conf_row["confidence"]
    )

    overall, size_df, crowded_df, gt_df = detailed_metrics(
        pred_dict,
        best_conf,
    )

    small_recall = value_for_size(
        size_df,
        "small",
    )

    medium_recall = value_for_size(
        size_df,
        "medium",
    )

    large_recall = value_for_size(
        size_df,
        "large",
    )

    crowded_recall = float(
        crowded_df.loc[
            crowded_df["crowded"] == True,
            "recall",
        ].iloc[0]
    )

    row = {
        "checkpoint": str(checkpoint_path),
        "checkpoint_name": checkpoint_path.name,

        "precision": float(overall["precision"]),
        "overall_recall": float(overall["recall"]),
        "f1": float(overall["f1"]),

        "small_recall": float(small_recall),
        "medium_recall": float(medium_recall),
        "large_recall": float(large_recall),
        "crowded_recall": float(crowded_recall),

        "TP": int(overall["TP"]),
        "FP": int(overall["FP"]),
        "FN": int(overall["FN"]),

        "best_conf": best_conf,

        "map50": map50,
        "map50_95": map50_95,

        "model_inference_ms": model_ms,
        "wall_ms_per_image": float(wall_ms),
    }

    print(f"mAP50:95      : {row['map50_95']:.4f}")
    print(f"mAP50         : {row['map50']:.4f}")
    print(f"Best conf     : {row['best_conf']:.3f}")
    print(f"Precision     : {row['precision']:.4f}")
    print(f"Overall Recall: {row['overall_recall']:.4f}")
    print(f"F1            : {row['f1']:.4f}")
    print(f"Small Recall  : {row['small_recall']:.4f}")
    print(f"Medium Recall : {row['medium_recall']:.4f}")
    print(f"Large Recall  : {row['large_recall']:.4f}")
    print(f"Crowded Recall: {row['crowded_recall']:.4f}")
    print(f"TP / FP / FN  : {row['TP']} / {row['FP']} / {row['FN']}")

    return row


stage41_rows = [
    audit_checkpoint(p)
    for p in unique_ckpts
]

stage41_df = pd.DataFrame(
    stage41_rows
)

AUDIT_CSV = (
    ANALYSIS41
    / "stage41_checkpoint_audit.csv"
)

stage41_df.to_csv(
    AUDIT_CSV,
    index=False,
)

print("\nSTATUS: STAGE41_CHECKPOINT_AUDIT_COMPLETE")


In [ ]:
# ============================================================
# CELL 10 — SELECT BEST CHECKPOINTS + FINAL COMPARISON
# ============================================================

best_map_row = stage41_df.loc[
    stage41_df["map50_95"].idxmax()
]

best_f1_row = stage41_df.loc[
    stage41_df["f1"].idxmax()
]

best_small_row = (
    stage41_df
    .sort_values(
        by=[
            "small_recall",
            "map50_95",
            "precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .iloc[0]
)


BEST_MAP_PATH = (
    ANALYSIS41
    / "best_stage41_map50_95.pt"
)

BEST_F1_PATH = (
    ANALYSIS41
    / "best_stage41_operational_f1.pt"
)

BEST_SMALL_PATH = (
    ANALYSIS41
    / "best_stage41_small_recall.pt"
)

shutil.copy2(
    best_map_row["checkpoint"],
    BEST_MAP_PATH,
)

shutil.copy2(
    best_f1_row["checkpoint"],
    BEST_F1_PATH,
)

shutil.copy2(
    best_small_row["checkpoint"],
    BEST_SMALL_PATH,
)


def row_to_ref(row, method):
    return {
        "method": method,
        "precision": float(row["precision"]),
        "overall_recall": float(row["overall_recall"]),
        "small_recall": float(row["small_recall"]),
        "medium_recall": float(row["medium_recall"]),
        "large_recall": float(row["large_recall"]),
        "crowded_recall": float(row["crowded_recall"]),
        "TP": int(row["TP"]),
        "FP": int(row["FP"]),
        "FN": int(row["FN"]),
        "best_conf": float(row["best_conf"]),
        "map50": float(row["map50"]),
        "map50_95": float(row["map50_95"]),
    }


map_ref = row_to_ref(
    best_map_row,
    "Stage4.1 Best mAP",
)

f1_ref = row_to_ref(
    best_f1_row,
    "Stage4.1 Best F1",
)

small_ref = row_to_ref(
    best_small_row,
    "Stage4.1 Best Small",
)


comparison = pd.DataFrame([
    stage2_ref,
    stage3_ref,
    stage4_ref,
    map_ref,
    f1_ref,
    small_ref,
])

COMPARISON_CSV = (
    ANALYSIS41
    / "stage2_stage3_stage4_stage41_comparison.csv"
)

comparison.to_csv(
    COMPARISON_CSV,
    index=False,
)


print("=" * 155)
print("FINAL YOLO COMPARISON")
print("=" * 155)

print(
    comparison.to_string(
        index=False,
        formatters={
            "precision": "{:.4f}".format,
            "overall_recall": "{:.4f}".format,
            "small_recall": "{:.4f}".format,
            "medium_recall": "{:.4f}".format,
            "large_recall": "{:.4f}".format,
            "crowded_recall": "{:.4f}".format,
            "best_conf": "{:.3f}".format,
            "map50": "{:.4f}".format,
            "map50_95": "{:.4f}".format,
        },
    )
)


print("\n" + "=" * 110)
print("BEST STAGE4.1 CHECKPOINTS")
print("=" * 110)

print("\n[A] Best mAP50:95")
print(best_map_row.to_string())

print("\n[B] Best Operational F1")
print(best_f1_row.to_string())

print("\n[C] Best Small Recall")
print(best_small_row.to_string())


print("\n" + "=" * 110)
print("STAGE3 → STAGE4.1 BEST-mAP DELTAS")
print("=" * 110)

for metric in [
    "precision",
    "overall_recall",
    "small_recall",
    "medium_recall",
    "large_recall",
    "crowded_recall",
    "map50",
    "map50_95",
]:
    print(
        f"{metric:<20}: "
        f"{float(map_ref[metric]) - float(stage3_ref[metric]):+.4f}"
    )


# Strong candidate = the same checkpoint must satisfy all.
strong_candidates = stage41_df[
    (stage41_df["small_recall"] >= 0.50)
    & (stage41_df["precision"] >= 0.75)
    & (stage41_df["overall_recall"] >= 0.72)
    & (stage41_df["large_recall"] >= 0.82)
    & (stage41_df["map50_95"] >= 0.45)
].copy()


print("\n" + "=" * 110)
print("SUCCESS CHECKS")
print("=" * 110)

print(
    "Any Small Recall >= 0.50 :",
    bool((stage41_df["small_recall"] >= 0.50).any()),
)

print(
    "Any mAP50:95 >= Stage3   :",
    bool((stage41_df["map50_95"] >= 0.473).any()),
)

print(
    "Any Precision >= 0.75    :",
    bool((stage41_df["precision"] >= 0.75).any()),
)

print(
    "Any Overall Recall >= .72:",
    bool((stage41_df["overall_recall"] >= 0.72).any()),
)

print(
    "Any Crowded Recall >= .70:",
    bool((stage41_df["crowded_recall"] >= 0.70).any()),
)

print(
    "Strong balanced candidates:",
    len(strong_candidates),
)

if len(strong_candidates):
    print(
        strong_candidates[
            [
                "checkpoint_name",
                "precision",
                "overall_recall",
                "f1",
                "small_recall",
                "medium_recall",
                "large_recall",
                "crowded_recall",
                "map50",
                "map50_95",
                "best_conf",
            ]
        ]
        .sort_values(
            ["map50_95", "small_recall"],
            ascending=False,
        )
        .to_string(index=False)
    )


summary = {
    "experiment": "YOLO11n_P2_Stage4_1_HybridRecovery",
    "starting_checkpoint": str(BEST_STAGE4),
    "additional_epochs": STAGE41_EPOCHS,
    "architecture": "YOLO11n-P2 P2/P3/P4/P5",
    "strides": stage41_strides,
    "imgsz": STAGE41_IMGSZ,

    "best_map_checkpoint": str(BEST_MAP_PATH),
    "best_f1_checkpoint": str(BEST_F1_PATH),
    "best_small_checkpoint": str(BEST_SMALL_PATH),

    "best_map_metrics": best_map_row.to_dict(),
    "best_f1_metrics": best_f1_row.to_dict(),
    "best_small_metrics": best_small_row.to_dict(),

    "strong_balanced_candidate_count": int(
        len(strong_candidates)
    ),

    "test_used": False,
}


def json_safe(obj):
    if isinstance(obj, dict):
        return {
            k: json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(obj, list):
        return [
            json_safe(v)
            for v in obj
        ]

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.bool_):
        return bool(obj)

    return obj


SUMMARY_JSON = (
    ANALYSIS41
    / "stage41_final_summary.json"
)

with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        json_safe(summary),
        f,
        indent=2,
        ensure_ascii=False,
    )


print("\nSaved:")
print("Best mAP   :", BEST_MAP_PATH)
print("Best F1    :", BEST_F1_PATH)
print("Best Small :", BEST_SMALL_PATH)
print("Audit CSV  :", AUDIT_CSV)
print("Comparison :", COMPARISON_CSV)
print("Summary    :", SUMMARY_JSON)

print("\nDetector TEST remains untouched.")
print("STATUS: STAGE4_1_HYBRID_RECOVERY_COMPLETE")


## What to send back after the run

Send the output starting from:

`STAGE 4.1 CHECKPOINT AUDIT`

through:

`SUCCESS CHECKS`

and the final:

`STATUS: STAGE4_1_HYBRID_RECOVERY_COMPLETE`

The main decision is whether one Stage-4.1 checkpoint can recover overall detection quality **and** improve Small Recall beyond Stage 3.
